<a href="https://colab.research.google.com/github/BariscanTosyali/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BariscanTosyali/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Starter dataset loaded successfully. Shape:", df.shape)

Starter dataset loaded successfully. Shape: (30000, 44)


Unit of Analysis: One row = One unique content page URL (content_id) aggregated over a 90-day search telemetry window.

Time Window & Panel Strategy: We evaluate historical performance using a mid-panel evaluation slice (2026-03 telemetry window) for model feature engineering and contract verification. We strictly treat the final month (2026-06) as a sealed, out-of-time test window to prevent temporal leakage during model development.

In [9]:
total_rows = len(df)
unique_pages = df["content_id"].nunique()
print(f"Total Rows: {total_rows}")
print(f"Unique Pages (content_id): {unique_pages}")
assert total_rows == unique_pages, "Grain violation: Row count does not match unique content_id count!"
print("Grain Verification Passed: Exactly 1 row = 1 unique page.")

Total Rows: 30000
Unique Pages (content_id): 30000
Grain Verification Passed: Exactly 1 row = 1 unique page.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Data Contract Field Classification:Features ($X$): content_age_days, days_since_last_update, impressions_90d, avg_position, ctr, word_count (Telemetry and content metadata knowable at the decision moment).Label ($y$): is_declining_label (Binary proxy target derived as 1 if trend_direction == "down", else 0).Context Columns: content_id, category (Identifiers and slicing metadata used for routing, grouping, and filtering).Excluded Columns: trend_pct (The direct percentage organic drop) and internal decision health flags.Why Excluded: Including these columns creates explicit data leakage because they measure the outcome after or during the decay process, making the model artificially perfect while failing on real unobserved data.

In [11]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

feature_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
label_col = "is_declining_label"
excluded_cols = ["trend_pct", "trend_direction"]

print("Features (X):", feature_cols)
print("Label (y):", label_col)
print("Excluded (Leakage Risk):", excluded_cols)

Features (X): ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Label (y): is_declining_label
Excluded (Leakage Risk): ['trend_pct', 'trend_direction']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Data Verification & Feature Availability Rationale:

Verification 1 (Grain Check): Exactly 1 row = 1 unique content_id.

Verification 2 (Row Count & Null Check): Verifying valid surviving rows using explicit IS TRUE / non-null filtering.

Verification 3 (Feature Availability): All 5 selected features (content_age_days, days_since_last_update, impressions_90d, avg_position, ctr) are knowable at the decision moment because they are calculated strictly from historical 90-day search logs prior to the intervention date.

Deliberate Leakage Experiment (The Trap): We intentionally add trend_pct to the feature set, observe an artificial jump in model accuracy/precision, and then purge it to recover an honest baseline score.

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Query 1: Availability Check (Surviving rows with valid telemetry)
surviving_rows = df[df["impressions_90d"].notna() & (df["impressions_90d"] > 0)]
print(f"Query 1 - Valid Telemetry Availability: {len(surviving_rows)} / {len(df)} rows survive IS TRUE check.")

# Query 2: Missing Value Audit
null_counts = df[feature_cols].isna().sum()
print("\nQuery 2 - Missing Value Audit per Feature:\n", null_counts)

# Query 3: Feature Range Verification
print("\nQuery 3 - Feature Summary Statistics:")
print(df[feature_cols].describe().T[["mean", "std", "min", "max"]])

# --- THE TRAP: Deliberate Feature Leakage Experiment ---
X_honest = df[feature_cols].fillna(0)
X_leaky = df[feature_cols + ["trend_pct"]].fillna(0)
y = df[label_col]

# Honest Model
clf_honest = RandomForestClassifier(random_state=42, max_depth=5)
clf_honest.fit(X_honest, y)
pred_honest = clf_honest.predict(X_honest)
honest_precision = precision_score(y, pred_honest)

# Leaky Model (The Trap)
clf_leaky = RandomForestClassifier(random_state=42, max_depth=5)
clf_leaky.fit(X_leaky, y)
pred_leaky = clf_leaky.predict(X_leaky)
leaky_precision = precision_score(y, pred_leaky)

print("\n--- LEAKAGE EXPERIMENT RESULTS ---")
print(f"Honest Model Precision: {honest_precision:.4f}")
print(f"Leaky Model Precision (WITH trend_pct TRAP): {leaky_precision:.4f}")
print("Purging 'trend_pct' to maintain an honest decision-support model score!")

Query 1 - Valid Telemetry Availability: 30000 / 30000 rows survive IS TRUE check.

Query 2 - Missing Value Audit per Feature:
 content_age_days             0
days_since_last_update       0
impressions_90d              0
avg_position                 0
ctr                          0
word_count                7699
dtype: int64

Query 3 - Feature Summary Statistics:
                               mean           std   min       max
content_age_days         256.167800    132.707930  90.0     564.0
days_since_last_update    46.098300     42.078709   1.0     373.0
impressions_90d         5200.366300  16838.019547   1.0  517715.0
avg_position              16.342380     15.216790   0.0     245.0
ctr                        0.510733      3.279162   0.0     100.0
word_count              3107.760325   1452.382598   8.0    9546.0

--- LEAKAGE EXPERIMENT RESULTS ---
Honest Model Precision: 0.6516
Leaky Model Precision (WITH trend_pct TRAP): 1.0000
Purging 'trend_pct' to maintain an honest decision-sup

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Data Limits & Boundary Constraints:

GSC Telemetry Latency: Google Search Console data carries a 2 to 3-day reporting lag; real-time real-hour traffic spikes cannot be captured at the exact moment of inference.

Unobserved Conversion/Revenue Outcomes: The warehouse slice contains search impression and click volume, but lacks downstream post-click conversion rates or revenue metrics.

Window Overlaps & Seasonality: A 90-day rolling aggregate smooths out seasonal micro-spikes (e.g., holiday trends), meaning short-term seasonal traffic bumps may temporarily mask long-term content decay.

In [15]:
# Veri sınırları ve aralık doğrulaması
print("Data Telemetry Limits & Range Summary:")
print(f"- Minimum Content Age: {df['content_age_days'].min()} days")
print(f"- Maximum Content Age: {df['content_age_days'].max()} days")
print(f"- Zero Impression Pages: {(df['impressions_90d'] == 0).sum()} pages (Unobserved/Dormant pages)")

Data Telemetry Limits & Range Summary:
- Minimum Content Age: 90 days
- Maximum Content Age: 564 days
- Zero Impression Pages: 0 pages (Unobserved/Dormant pages)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.